# KNOWLY 分析ノートブック②　出題方式を公平に比べる

ノート①では、ランダム出題と SM-2 を比べました。このノートでは、**生徒ごとの記録から「いま正解できる確率」を予測して出題を決める方式**(個人適応方式)を作って、比べます。
ノート①のシミュレーションを入れ直した、**これ1つで最後まで動くノートブック**です。ノート①の結果は、このノートに置き換えます(理由は「3. 出題方式」に書きました)。

| 方式 | 出し方 |
|---|---|
| ランダム | 毎日、問題をランダムに選ぶ |
| SM-2 | 全員に同じ間隔で、期限が来た問題を出す |
| 個人適応(低い順) | 予測が低い問題、つまり**いちばん忘れた問題**から出す(最初の案) |
| 個人適応(直前) | 予測が目標に近い問題、つまり**忘れる直前の問題**から出す(改良案) |
| 個人適応(直前+再挑戦) | 「直前」に、**間違えた問題をもう一度出す枠**を足したもの(放置を減らす改良) |
| 理想(上限) | 「直前」と同じルールで、予測の代わりに**本当の覚えの強さ**を使う版。本物のアプリでは作れない。「予測を完璧にしたら、どこまで伸びるか」を測る物差し(再挑戦など、別のルールは入っていないので、これを超える方式もあります) |

**注意**:使うのは、仮のルールで作った架空のデータです。結果は「仕組みの確認」と「考え方の比較」で、「効果の証明」ではありません。

## 1. 準備と設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pathlib import Path

# グラフで日本語を表示するための設定(Windowsに入っているフォントを順に探す)
plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ---- 設定(ここを変えると結果が変わります) ----
N_STUDENTS = 20      # 生徒の人数
N_QUESTIONS = 50     # 問題の数(理科)
STUDY_DAYS = 30      # 学習する日数
PER_DAY = 10         # 1日に出す問題数
NEW_RATIO = 0.2      # 1日の出題のうち、新しい問題にあてる割合(すべての問題に触れるため)
RETRY_RATIO = 0.2    # 1日の出題のうち、間違えた問題の再挑戦にあてる割合(再挑戦つきの方式だけ)
REST_DAYS = 14       # 学習をやめてから、最終テストまであける日数
SEED = 42            # 乱数の種(同じ値なら、毎回同じ結果になります)
START_DATE = datetime(2026, 11, 1, 18, 0)   # 学習を始める日時(架空)

DATA_DIR = Path("data")      # 作ったデータの保存先
DATA_DIR.mkdir(exist_ok=True)

## 2. 架空の生徒の「忘れ方」(ノート①と同じ)

覚えている確率は、忘却曲線で決まります。

> 正解できる確率 = 勘で当たる確率 + (1 − 勘で当たる確率) × exp( −経過日数 ÷ S )

S は「覚えの強さ」(日数)で、生徒の得意さと問題の難しさで決まります。
思い出せたときは S が伸び(忘れかけで思い出せたほど大きく)、思い出せなかったときは S が縮みます。

**大事な点**:この「本当の S」は、シミュレーションの中だけにある答えです。出題方式からは見えません。
本物のアプリでも、生徒の頭の中は見えず、見えるのは「解答の記録」だけです。

In [ ]:
rng_master = np.random.default_rng(SEED)
ability = rng_master.normal(0, 1, N_STUDENTS)       # 生徒の得意さ(大きいほど得意)
difficulty = rng_master.normal(0, 1, N_QUESTIONS)   # 問題の難しさ(大きいほど難しい)

# 初めて見たときの「覚えの強さ」(日数)。形は(生徒の数, 問題の数)
S0 = 1.5 * np.exp(0.5 * ability[:, None] - 0.5 * difficulty[None, :])

PRIOR = 0.2   # 勘や知識で当てられる確率(一度も見ていない問題も、忘れた問題もこの値)


def recall_prob(S, elapsed_days):
    """経過日数から、いまその問題に正解できる確率を返す(忘却曲線)。勘で当たる分(PRIOR)が下限"""
    return PRIOR + (1 - PRIOR) * np.exp(-elapsed_days / S)


def new_state():
    """記憶の状態を作る。S=覚えの強さ、last=最後に解いた日(nan=まだ解いていない)"""
    return {
        "S": S0.copy(),
        "last": np.full((N_STUDENTS, N_QUESTIONS), np.nan),
    }


def answer(state, i, q, day, rng):
    """生徒 i が問題 q に day 日目に答える。(正誤, 解答秒数)を返し、記憶の状態も更新する"""
    last = state["last"][i, q]
    if np.isnan(last):
        p = PRIOR                                         # 初見
    else:
        p = recall_prob(state["S"][i, q], day - last)     # 正解できる確率
    correct = bool(rng.random() < p)

    # 覚えているほど早く答えられる(ばらつきも足す)
    sec = float(np.clip(rng.normal(14 - 9 * p, 2.5), 2, 40))

    # 解答のあとで、記憶の強さを更新する
    if np.isnan(last):
        pass                                              # 初見は、答えを見ただけ(S0 のまま)
    elif correct:
        state["S"][i, q] *= 1.2 + 2.0 * (1 - p)           # 忘れかけで思い出せたほど強く定着
    else:
        state["S"][i, q] = max(0.5 * S0[i, q], 0.4 * state["S"][i, q])
    state["last"][i, q] = day
    return correct, sec


def mean_recall(state, day):
    """day 日目の朝の時点で、全員・全問題の「正解できる確率」の平均と、生徒ごとの平均を返す"""
    elapsed = day - state["last"]
    R = np.where(np.isnan(state["last"]), PRIOR, recall_prob(state["S"], elapsed))
    return R.mean(), R.mean(axis=1)

## 3. 出題方式

どの方式も、同じ2つの動きをします。

- `pick(...)`:今日の問題を選ぶ
- `update(...)`:解答の結果を受け取って、次に活かす

方式からは、解答の「記録」(いつ、どの問題を、正解/不正解で、何秒で)しか見えません。

### 公平に比べるためのルール:新しい問題の枠

最初の試作では、SM-2 などが「復習を優先しすぎて、新しい問題をあまり出さない」状態になっていました。
実際に、1人が触れた問題が50問中30問ほどにとどまっていました。
最終日の点数は「全50問の平均」なので、一部の問題だけを深く覚える方式が、有利に見えてしまいます。(**ノート①の比べ方には、この問題がありました。**)

そこで、**SM-2 と個人適応では、1日の出題のうち一定の割合(`NEW_RATIO`、初期設定は20%=10問中2問)を、まだ解いていない問題にあてる**ことにしました。
ランダム方式は、もともと全問題から選ぶので、そのままです。あとで、各方式が何問に触れたかも確認します。

### 個人適応方式のしくみ

1. 問題ごと・生徒ごとに、「覚えの強さ」を推定した値(`S_hat`)を持ちます。最初は全員・全問題で同じ仮の値(2日)です。
2. 解答のたびに、`S_hat` を更新します。正解なら大きく(すぐ答えられたら、さらに大きく)、不正解なら小さくします。
3. 生徒ごとに「忘れにくさ」の係数 `k` を持ちます(最初は1)。予想より正解できていれば `k` を上げ、外れていれば下げます。
4. 「S_hat × k」から、各問題の「いま正解できる確率」を予測します。
5. 予測が目標(`target`)を下回った問題を「復習の対象」にして、その中から出します。

**ここで、出す順番を2通り試します。**

- **低い順**(最初の案):予測がいちばん低い問題、つまり、いちばん忘れた問題から出す。
- **直前**(改良案):予測が目標にいちばん近い問題、つまり、忘れる直前の問題から出す。

### 放置を減らす改良:再挑戦枠

「直前」方式は、一度間違えた問題の予測が下がると、選ばれにくくなり、**苦手な問題が放置されやすい**ことが分かりました(ノート③で確認)。
そこで、**1日の出題のうち一定の割合(`RETRY_RATIO`、初期設定は20%=10問中2問)を、「前回間違えた問題」の再挑戦にあてる**方式を足しました。
間違えてから時間がたっている問題から順に出します。正解できたら、再挑戦の対象から外れます。
(出題数が5問未満のときは、再挑戦枠は0になります。)

「理想」は、「直前」と同じルールで、予測の代わりに本当の確率を使う方式です。本物のアプリでは作れませんが、**予測を完璧にしたら、どこまで伸びるか**を知るための物差しになります。
(「上限」と名前がついていますが、再挑戦のような別のルールは入っていないので、**これを超える方式もありえます**。)

In [ ]:
def new_slots():
    """1日の出題のうち、新しい問題にあてる数(最低1問)"""
    return max(1, round(PER_DAY * NEW_RATIO))


def retry_slots():
    """1日の出題のうち、間違えた問題の再挑戦にあてる数(出題数が5問未満なら0)"""
    return int(PER_DAY * RETRY_RATIO)


def assemble(reviews, unseen, rest, retry=()):
    """今日の問題を組み立てる。新しい問題の枠、再挑戦の枠を先に確保して、残りを復習で埋める。
    reviews=復習する問題(優先順)、unseen=まだ解いていない問題(順番はランダム)、
    rest=余ったときの予備(優先順)、retry=前回間違えた問題(優先順)"""
    chosen = list(unseen[:min(new_slots(), len(unseen))])
    chosen += [q for q in retry if q not in chosen][:max(0, min(retry_slots(), PER_DAY - len(chosen)))]
    chosen += [q for q in reviews if q not in chosen][:PER_DAY - len(chosen)]
    if len(chosen) < PER_DAY:      # 復習する問題が足りなければ、新しい問題を足す
        chosen += [q for q in unseen if q not in chosen][:PER_DAY - len(chosen)]
    if len(chosen) < PER_DAY:      # それでも足りなければ、予備から
        chosen += [q for q in rest if q not in chosen][:PER_DAY - len(chosen)]
    return chosen


class RandomPolicy:
    """ランダム方式:問題をランダムに選ぶ"""

    def pick(self, state, i, day, rng):
        return [int(q) for q in rng.choice(N_QUESTIONS, PER_DAY, replace=False)]

    def update(self, i, q, day, correct, sec):
        pass


class SM2Policy:
    """SM-2方式(簡易版):問題ごとに「次に出す日」を決め、期限が来た問題を出す"""

    def __init__(self):
        shape = (N_STUDENTS, N_QUESTIONS)
        self.due = np.full(shape, np.inf)       # 次に出す日(inf=まだ解いていない)
        self.reps = np.zeros(shape, dtype=int)  # 連続正解の回数
        self.ease = np.full(shape, 2.5)         # 易しさ
        self.interval = np.zeros(shape)         # 間隔(日)

    def pick(self, state, i, day, rng):
        due = self.due[i]
        seen = np.isfinite(due)
        order = [int(q) for q in np.argsort(due)]                    # 期限が近い順
        reviews = [q for q in order if seen[q] and due[q] <= day]    # 期限が来た問題
        unseen = [int(q) for q in rng.permutation(N_QUESTIONS) if not seen[q]]
        return assemble(reviews, unseen, [q for q in order if seen[q]])

    def update(self, i, q, day, correct, sec):
        if not correct:
            self.reps[i, q] = 0
            self.interval[i, q] = 1
        else:
            quality = 5 if sec < 8 else 4        # すぐ答えられたら5、時間がかかったら4
            self.reps[i, q] += 1
            if self.reps[i, q] == 1:
                self.interval[i, q] = 1
            elif self.reps[i, q] == 2:
                self.interval[i, q] = 6
            else:
                self.interval[i, q] = round(self.interval[i, q] * self.ease[i, q])
            e = self.ease[i, q] + 0.1 - (5 - quality) * (0.08 + (5 - quality) * 0.02)
            self.ease[i, q] = max(1.3, e)
        self.due[i, q] = day + self.interval[i, q]


S_INIT = 2.0   # 個人適応方式の、「覚えの強さ」の最初の推定値(日)。全員・全問題で同じ仮の値


class PersonalPolicy:
    """個人適応方式:「いま正解できる確率」を予測して、復習する問題を選ぶ"""

    def __init__(self, target=0.8, lr=0.15, order="near", retry=False):
        shape = (N_STUDENTS, N_QUESTIONS)
        self.target = target                    # この確率を下回った問題を「復習の対象」とする
        self.retry = retry                      # True なら、間違えた問題の再挑戦枠をつくる
        self.wrong = np.zeros(shape, dtype=bool)  # 前回の解答が不正解だったか
        self.lr = lr                            # 生徒の係数 k を、1回でどれだけ動かすか
        self.order = order                      # "near"=目標に近い順(忘れる直前) / "low"=確率が低い順
        self.S_hat = np.full(shape, S_INIT)     # 問題ごとの「覚えの強さ」の推定
        self.k = np.ones(N_STUDENTS)            # 生徒ごとの「忘れにくさ」の推定(1が平均)
        self.last = np.full(shape, np.nan)      # 最後に解いた日

    def predict(self, i, day):
        """生徒 i の全問題について、今日 正解できる確率を予測する"""
        elapsed = day - self.last[i]
        p = recall_prob(self.S_hat[i] * self.k[i], elapsed)
        return np.where(np.isnan(self.last[i]), PRIOR, p)

    def pick(self, state, i, day, rng):
        p = self.predict(i, day)
        seen = ~np.isnan(self.last[i])
        sort_key = -p if self.order == "near" else p
        order = [int(q) for q in np.argsort(sort_key)]
        reviews = [q for q in order if seen[q] and p[q] < self.target]   # 復習の対象
        unseen = [int(q) for q in rng.permutation(N_QUESTIONS) if not seen[q]]
        retry = []
        if self.retry:   # 前回間違えた問題を、間があいている順に
            retry = [int(q) for q in np.argsort(self.last[i]) if seen[q] and self.wrong[i, q] and self.last[i, q] < day]
        return assemble(reviews, unseen, [q for q in order if seen[q]], retry)

    def update(self, i, q, day, correct, sec):
        self.wrong[i, q] = not correct
        if not np.isnan(self.last[i, q]):
            # 予測と結果のずれから、生徒の係数 k を更新する
            p_pred = recall_prob(self.S_hat[i, q] * self.k[i], day - self.last[i, q])
            self.k[i] = float(np.clip(self.k[i] * np.exp(self.lr * (float(correct) - p_pred)), 0.3, 3.0))
            # 問題ごとの「覚えの強さ」の推定を更新する
            if correct:
                self.S_hat[i, q] *= 2.5 if sec < 8 else 1.8   # すぐ答えられたら、より強く
            else:
                self.S_hat[i, q] = max(0.5 * S_INIT, 0.5 * self.S_hat[i, q])
        self.last[i, q] = day


class OraclePolicy:
    """理想(上限):本当の覚えの強さを知っている方式。診断用で、本物のアプリでは作れない"""

    def __init__(self, target=0.8):
        self.target = target

    def pick(self, state, i, day, rng):
        last = state["last"][i]
        seen = ~np.isnan(last)
        p = np.where(seen, recall_prob(state["S"][i], day - np.nan_to_num(last)), PRIOR)
        order = [int(q) for q in np.argsort(-p)]                         # 目標に近い順
        reviews = [q for q in order if seen[q] and p[q] < self.target]
        unseen = [int(q) for q in rng.permutation(N_QUESTIONS) if not seen[q]]
        return assemble(reviews, unseen, [q for q in order if seen[q]])

    def update(self, i, q, day, correct, sec):
        pass

## 4. シミュレーションを動かす

30日間の学習のあと、14日あけて、最終日(44日目)の「正解できる確率」を測ります。
解答の記録は、設計メモで決めた6項目の形で残します。

In [ ]:
LOG_COLUMNS = ["student_id", "question_id", "answered_at", "correct", "response_sec", "attempt_no"]
TEST_DAY = STUDY_DAYS + REST_DAYS


def run_policy(make_policy, seed):
    """1つの出題方式で、学習から最終テストまでを動かす。(記録, 毎朝の平均, 生徒ごとの最終結果, 方式)を返す"""
    rng = np.random.default_rng(seed)
    state = new_state()
    policy = make_policy()
    attempts = np.zeros((N_STUDENTS, N_QUESTIONS), dtype=int)
    rows = []
    daily = []                          # 毎朝の「正解できる確率」の平均

    for day in range(TEST_DAY + 1):
        mean_all, _ = mean_recall(state, day)
        daily.append(mean_all)
        if day >= STUDY_DAYS:           # 学習をやめたあとは、測るだけ
            continue
        for i in range(N_STUDENTS):
            offset = 0.0                # その日の学習を始めてからの経過秒数
            for q in policy.pick(state, i, day, rng):
                correct, sec = answer(state, i, q, day, rng)
                attempts[i, q] += 1
                rows.append((
                    f"S{i + 1:03d}",
                    f"Q{q + 1:03d}",
                    START_DATE + timedelta(days=day, minutes=i * 5, seconds=round(offset)),
                    int(correct),
                    round(sec, 1),
                    int(attempts[i, q]),
                ))
                offset += sec + 3       # 解答時間 + 次の問題へ進む時間
                policy.update(i, q, day, correct, sec)

    log = pd.DataFrame(rows, columns=LOG_COLUMNS)
    _, final_by_student = mean_recall(state, TEST_DAY)
    return log, np.array(daily), final_by_student, policy


def coverage(log):
    """1人あたり、何問に触れたか(50問中)"""
    return log.groupby("student_id")["question_id"].nunique().mean()


def neglect_rate(log):
    """放置率:1人が、その問題を2回以下しか解かなかった割合(復習が足りない問題の割合)"""
    counts = log.groupby(["student_id", "question_id"]).size().unstack(fill_value=0).values
    return float((counts <= 2).mean())


# 動かす方式の一覧(target は、あとで選び直します)
def make_methods(target, retry_target=None):
    retry_target = target if retry_target is None else retry_target
    return {
        "ランダム": RandomPolicy,
        "SM-2": SM2Policy,
        "個人適応(低い順)": lambda: PersonalPolicy(target=target, order="low"),
        "個人適応(直前)": lambda: PersonalPolicy(target=target, order="near"),
        "個人適応(直前+再挑戦)": lambda: PersonalPolicy(target=retry_target, order="near", retry=True),
        "理想(上限)": lambda: OraclePolicy(target=target),
    }


results = {}
for name, make in make_methods(0.8).items():
    log, daily, final, policy = run_policy(make, SEED)
    results[name] = {"log": log, "daily": daily, "final": final, "policy": policy}

for name, r in results.items():
    print(f"{name:12s} 記録 {len(r['log'])}行   最終日の平均 {r['final'].mean() * 100:.1f}%")
results["個人適応(直前)"]["log"].head(8)

## 5. 設定値を選ぶ(目標の確率)

個人適応方式には、「予測が何%を下回ったら、復習の対象にするか」という設定(`target`)があります。
どの値がよいかを、**調整用の乱数(100〜104)** で試して選びます。
そのあとの比較は、**別の乱数(0〜19)** で行います。選ぶのに使ったデータで比べると、都合のよい結果になってしまうからです。

**ただし、点数が高い順に選ぶのは危険です。** `target` を高くするほど、「復習する問題を絞って、ほかの問題を放置する」出し方になり、点数が上がることが分かりました。
(「全50問の平均」という指標は、一部の問題を捨てても、点数が上がるためです。)
そこで、**放置率(2回以下しか解かなかった問題の割合)が 25% 以下のものの中で、いちばん点数が高いもの**を選びます。
(25% は、私が決めた目安です。変えると、選ばれる値も変わります。)

In [ ]:
NEGLECT_LIMIT = 0.25      # 放置率の上限(2回以下しか解かなかった問題が、全体の25%まで)
TUNE_SEEDS = range(100, 105)
TARGET_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]


def choose_target(make_with_target):
    """target の候補を順に試して、放置率が上限以下の中で、いちばん点数が高いものを選ぶ"""
    rows = []
    for target in TARGET_GRID:
        runs = [run_policy(lambda: make_with_target(target), s) for s in TUNE_SEEDS]
        rows.append((
            target,
            np.mean([r[2].mean() * 100 for r in runs]),
            np.mean([neglect_rate(r[0]) for r in runs]),
        ))
    table = pd.DataFrame(rows, columns=["target", "最終日の平均(%)", "放置率"])
    ok = table[table["放置率"] <= NEGLECT_LIMIT]
    best = float(ok.loc[ok["最終日の平均(%)"].idxmax(), "target"])
    return best, table.round({"target": 2, "最終日の平均(%)": 1, "放置率": 2})


BEST_TARGET, tune_df = choose_target(lambda t: PersonalPolicy(target=t, order="near"))
BEST_RETRY_TARGET, tune_retry_df = choose_target(lambda t: PersonalPolicy(target=t, order="near", retry=True))

print(f"個人適応(直前)       で選んだ target: {BEST_TARGET}")
display(tune_df)
print(f"個人適応(直前+再挑戦) で選んだ target: {BEST_RETRY_TARGET}")
display(tune_retry_df)
print(f"(放置率が {NEGLECT_LIMIT:.0%} 以下の中で、いちばん点数が高いものを選びました)")

### 点数と放置率の関係

`target` を高くすると、点数は上がりますが、放置率も上がります。**点数と放置率は、あちらを立てればこちらが立たない関係(トレードオフ)** です。
下の図は、`target` を変えながら、2つの方式の「点数」と「放置率」を並べたものです。
**左上にあるほど、よい方式**(放置が少なくて、点数が高い)です。再挑戦枠で、この線が左上に動いているかを見ます。

In [ ]:
plt.figure(figsize=(7, 4.4))
for table, label, color in [(tune_df, "個人適応(直前)", "tab:orange"),
                            (tune_retry_df, "個人適応(直前+再挑戦)", "tab:red")]:
    plt.plot(table["放置率"] * 100, table["最終日の平均(%)"], marker="o", label=label, color=color)
    for _, r in table.iterrows():
        plt.annotate(f"{r['target']:.2f}", (r["放置率"] * 100, r["最終日の平均(%)"]),
                     textcoords="offset points", xytext=(4, -10), fontsize=7)
plt.axvline(NEGLECT_LIMIT * 100, color="black", linestyle="--", linewidth=0.8)
plt.text(NEGLECT_LIMIT * 100 + 0.3, plt.ylim()[0] + 0.3, "放置率の上限", fontsize=8)
plt.xlabel("放置率(%)  ← 左ほど、放置が少ない")
plt.ylabel("最終日の正解できる確率(%)  ↑ 上ほど、高得点")
plt.title("点数と放置率の関係(点の横の数字は target)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. 方式を比べる

選んだ設定で、**調整に使っていない乱数(0〜19)** を使って、20回くり返して比べます。
点数だけでなく、**触れた問題数**と**放置率**(2回以下しか解かなかった問題の割合)も並べて、偏りがないかを見ます。

In [ ]:
REPEATS = 20
makers = make_methods(BEST_TARGET, BEST_RETRY_TARGET)

# 選んだ target で、乱数 42 の1回分を動かし直す(グラフ用)
results = {}
for name, make in makers.items():
    log, daily, final, policy = run_policy(make, SEED)
    results[name] = {"log": log, "daily": daily, "final": final, "policy": policy}

# 記録を、CSVで保存する(Excelで開いても文字化けしないようにする)。ノート③で分析します
results["個人適応(直前)"]["log"].to_csv(DATA_DIR / "log_personal.csv", index=False, encoding="utf-8-sig")
results["個人適応(直前+再挑戦)"]["log"].to_csv(DATA_DIR / "log_personal_retry.csv", index=False, encoding="utf-8-sig")

scores = {name: [] for name in makers}
covs = {name: [] for name in makers}
negs = {name: [] for name in makers}
for seed in range(REPEATS):
    for name, make in makers.items():
        log, _, final, _ = run_policy(make, seed)
        scores[name].append(final.mean() * 100)
        covs[name].append(coverage(log))
        negs[name].append(neglect_rate(log))
scores = {name: np.array(v) for name, v in scores.items()}

summary_df = pd.DataFrame({
    "最終日の平均(%)": {n: v.mean() for n, v in scores.items()},
    "ばらつき(±)": {n: v.std() for n, v in scores.items()},
    "触れた問題数(50問中)": {n: np.mean(v) for n, v in covs.items()},
    "放置率": {n: np.mean(v) for n, v in negs.items()},
})

# グラフ
colors = {"ランダム": "tab:gray", "SM-2": "tab:blue", "個人適応(低い順)": "tab:brown",
          "個人適応(直前)": "tab:orange", "個人適応(直前+再挑戦)": "tab:red", "理想(上限)": "tab:green"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

ax = axes[0]
days = np.arange(TEST_DAY + 1)
for name, r in results.items():
    ax.plot(days, r["daily"] * 100, label=name, color=colors[name])
ax.axvline(STUDY_DAYS, color="black", linestyle="--", linewidth=0.8)
ax.text(STUDY_DAYS + 0.5, 0.03, "学習をやめる", fontsize=9, transform=ax.get_xaxis_transform())
ax.set_xlabel("日数")
ax.set_ylabel("正解できる確率の平均(%)")
ax.set_title(f"正解できる確率の推移(乱数 {SEED} の1回分)")
ax.legend(fontsize=8)

ax = axes[1]
bp = ax.boxplot([scores[n] for n in scores], tick_labels=[n.replace("(", "\n(") for n in scores], patch_artist=True)
for patch, name in zip(bp["boxes"], scores):
    patch.set_facecolor(colors[name])
    patch.set_alpha(0.5)
ax.set_ylabel("最終日の正解できる確率(%)")
ax.set_title(f"最終日の結果({REPEATS}回くり返し)")
ax.tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.show()

summary_df.round({"最終日の平均(%)": 1, "ばらつき(±)": 1, "触れた問題数(50問中)": 1, "放置率": 2})

## 7. 「生徒ごとの補正(k)」は効いているか

個人適応方式の特徴は、生徒ごとの係数 `k` で予測を補正することです。本当に効いているかを、2つの方法で確かめます。

1. `k` を使わない版(`lr=0`)と比べる。
2. 推定した `k` が、本当の得意さ(シミュレーションだけで分かる)と合っているかを見る。

In [ ]:
with_k = scores["個人適応(直前)"]
without_k = np.array([
    run_policy(lambda: PersonalPolicy(target=BEST_TARGET, lr=0.0, order="near"), s)[2].mean() * 100
    for s in range(REPEATS)
])
diff = with_k - without_k
print(f"{REPEATS}回くり返した、最終日の平均")
print(f"  k あり(個人適応): {with_k.mean():.1f}%")
print(f"  k なし            : {without_k.mean():.1f}%")
print(f"  差(あり − なし)  : {diff.mean():+.1f} ポイント(k ありの方が高かった回数: {REPEATS}回中 {(diff > 0).sum()}回)")

# 推定した k と、本当の得意さの比較
k = results["個人適応(直前)"]["policy"].k
corr = pd.Series(k).corr(pd.Series(ability), method="spearman")
print(f"推定した k と、本当の得意さの順位相関: {corr:.2f}   (1に近いほど、順位がよく合っている)")

plt.figure(figsize=(5.5, 4))
plt.scatter(ability, k, color="tab:orange")
plt.xlabel("本当の得意さ(シミュレーションだけで分かる)")
plt.ylabel("推定した係数 k")
plt.title("生徒ごとの推定は、当たっているか")
plt.tight_layout()
plt.show()

## 8. 1日の出題数を変えると、どうなるか

出題数を変えて、方式を比べます。(調整用ではない乱数 0〜5 を使います。)

In [ ]:
original_per_day = PER_DAY
sweep = []
for n in [2, 3, 5, 10, 20]:
    PER_DAY = n                       # 1日の出題数を変えて、6回ずつくり返す
    row = {"1日の出題数": n}
    for name, make in makers.items():
        row[name] = np.mean([run_policy(make, s)[2].mean() * 100 for s in range(6)])
    sweep.append(row)
PER_DAY = original_per_day            # 元の設定に戻す

sweep_df = pd.DataFrame(sweep)

plt.figure(figsize=(7, 4))
for name in makers:
    plt.plot(sweep_df["1日の出題数"], sweep_df[name], marker="o", label=name, color=colors[name])
plt.xticks(sweep_df["1日の出題数"])
plt.xlabel("1日の出題数")
plt.ylabel("最終日の正解できる確率(%)")
plt.title("出題数を変えたときの比較")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

sweep_df.round(1)

## 9. まとめと、次にやること

**私が動かしたときの結果**(初期設定、乱数 0〜19 の20回。細かい数字は、環境によって少しずれることがあります)

| 方式 | 最終日の平均 | 放置率 |
|---|---|---|
| ランダム | 約27% | 約5% |
| SM-2 | 約24% | 約2% |
| 個人適応(低い順) | 約23% | 約7% |
| 個人適応(直前) | 約31% | 約20% |
| 個人適応(直前+再挑戦) | 約34% | 約22% |
| 理想(上限) | 約34% | 約25% |

**分かったこと**

- **出す順番が、結果を大きく左右しました。** 最初の案(いちばん忘れた問題から出す)は、ランダムにも SM-2 にも負けました。
  ほとんど忘れた問題は、出しても間違えやすく、間違えると覚えの強さが縮んでしまうためです。「忘れる直前の問題から出す」に変えると、ランダムを上回りました。
- **再挑戦枠は、「同じ放置率なら、点数が高い」という形で効きました。** 点数と放置率の図で、再挑戦つきの線が、左上に動いています。
  たとえば、放置率が2割前後の設定を比べると、「直前」は約31%、「直前+再挑戦」は約34%でした。
  同じ `target`(0.8)で比べると、再挑戦つきは、点数が約1.6ポイント低い代わりに、放置率が約19%から約14%に下がります。
  **何を優先するかで、選ぶ設定が変わります。**
- **再挑戦つきは、出題数が2〜20問のどの条件でも、ランダムを上回りました。** 「直前」だけだと、1日20問のときにランダムに負けていたので、改善しています。
- **「理想」を上回る条件もありました**(1日20問など)。「理想」は、再挑戦のルールを持たない方式に、本当の値を与えたものなので、全方式の上限ではありません。
- **生徒ごとの補正(k)は、少しだけ効きました。** k なしより約1ポイント高く、20回中20回で上回りました。ただ、推定した k と本当の得意さの順位相関は 0.44 で、まだ当たりきっていません。

**読み方と注意点**

- これは、仮の忘却モデルでの比較です。数字を変えれば、結果も変わります。「効果の証明」ではありません。
- **放置は、なくなってはいません。** 再挑戦をつけても、放置率は1〜2割あります。本物の生徒に使うときは、苦手な問題が放置されていないか、必ず確認する必要があります(ノート③で、記録から確認できます)。
- 放置率の上限(25%)、再挑戦の割合(20%)、新しい問題の割合(20%)は、私が決めた目安です。変えると、選ばれる設定も変わります。
- ノート①では、新しい問題の枠がなく、SM-2 が不公平に低くなる可能性がありました。比較は、このノートの結果に置き換えてください。

**次にやること**

- [ ] ノート③で、「直前」と「直前+再挑戦」の記録を比べて、放置された問題の様子が変わったかを確かめる
- [ ] 推定を良くする。`S_hat` の更新に使っている数字(2.5、1.8、0.5)は、いまは仮の値です。記録から、当てはまりのよい値を探す
- [ ] 機械学習(scikit-learn)で「次に正解できるか」を予測するモデルを作って、いまの簡単な推定と比べる
- [ ] 本物のデータが取れたら、同じ手順で比べ直す